[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

*Attention & Transformers*
Implement a **top-k Mixture of Experts** layer.

A router scores every expert per token, the top $k$ win, and their outputs are
combined with softmax weights over just those $k$ scores.

### Signature
```python
class MixtureOfExperts(nnx.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2, *, rngs: nnx.Rngs): ...
    def __call__(self, x): ...
```

### Requirements
- `self.router`: `nnx.Linear(d_model, num_experts)`
- `self.experts`: an **`nnx.List`** of `num_experts` MLPs, each
  `nnx.Sequential(nnx.Linear(d_model, d_ff), jax.nn.relu, nnx.Linear(d_ff, d_model))`
  (`nnx.List` is Flax's `nn.ModuleList` — a bare Python list is rejected)
- `self.top_k`
- Accepts `(B, S, D)` or `(N, D)`, and returns the same shape
- Routing is **per token**, not per sequence
- Softmax over the top-k logits **only**

### The point: parameters and FLOPs come apart
A dense layer uses every parameter for every token. An MoE with $E$ experts
holds $E\times$ the parameters but activates only $k$ of them per token, so
capacity grows while per-token compute stays fixed. Mixtral-8x7B has ~47B
parameters and the forward cost of a ~13B model, because $k=2$ of $8$ experts
run per token.

### Softmax over the top-k, not all E
Softmax first and then truncate, and the kept weights no longer sum to 1 — the
layer's output magnitude then depends on how confident the router happened to
be, which destabilises training. Select first, softmax second.

### Why real implementations need a load-balancing loss
Routing is a winner-take-all feedback loop: an expert that is slightly better
early gets more tokens, trains faster, and gets picked even more, until a few
experts do everything and the rest are dead weight. Production MoEs add an
auxiliary loss pushing the router toward uniform expert usage. This task leaves
it out to stay close to the original — but "what stops the router collapsing?"
is the follow-up question this problem exists to set up.

### ⚠️ Why the JAX version has no boolean-mask scatter
The PyTorch original writes `output[mask] += ...` with a boolean mask. JAX has
no in-place scatter and cannot handle a data-dependent output shape under
`jit`, so instead every expert runs on every token and its contribution is
multiplied by a per-token weight that is **zero** where the expert was not
selected. Same result; it is dense rather than sparse, which is fine at this
scale and is exactly why real sparse MoE needs custom kernels to actually
realise the FLOP saving.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class MixtureOfExperts(nnx.Module):
    """Top-k routed mixture of expert MLPs."""

    def __init__(self, d_model: int, d_ff: int, num_experts: int,
                 top_k: int = 2, *, rngs: nnx.Rngs):
        pass  # Replace this

    def __call__(self, x):
        """(B, S, d_model) or (N, d_model) -> same shape"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

moe = MixtureOfExperts(d_model=16, d_ff=32, num_experts=4, top_k=2,
                       rngs=nnx.Rngs(params=0))
x = jax.random.normal(jax.random.key(1), (2, 5, 16))
print("out:", moe(x).shape)

logits = moe.router(x.reshape(-1, 16))
_, idx = jax.lax.top_k(logits, 2)
counts = jnp.bincount(idx.ravel(), length=4)
print("tokens routed to each expert:", counts, "(uneven — hence the aux loss)")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("moe")

# hint("moe")      # stuck? nudge without the answer
# solution("moe")  # spoiler: the reference implementation